# 📑 RELATÓRIO TÉCNICO: ANÁLISE EXPLORATÓRIA E TRATAMENTO DE DADOS (EDA)

## Tabela: `silver.order_items`

**Projeto:** E-commerce ML Olist

**Autora:** Isabella 

---

## 1. Introdução e Contexto do Negócio

No desenho da nossa **Arquitetura Analítica**, a tabela `order_items` é o coração transacional do projeto. Ela registra a abertura de cada item vendido no ecossistema Olist.

Para atender ao objetivo estratégico de **Segmentação de Clientes VIP e Risco de Churn**, os dados desta tabela são fundamentais para calcular variáveis de comportamento de compra (Feature Engineering), como o gasto total do cliente, a quantidade de itens comprados e o valor médio do frete pago.

Este documento detalha o estado dos dados após a transição da camada **Bronze para a Silver**, validando as regras de limpeza, a eliminação de duplicados e o tratamento estatístico de outliers que aplicamos.

---

## 2. Dicionário de Dados (Contrato da Camada Silver)

Após o processamento do script SQL e as validações em Python, a estrutura da tabela ficou definida da seguinte forma:

| Nome da Coluna | Tipo de Dado | Descrição do Negócio | Regra de Limpeza / Silver |
| --- | --- | --- | --- |
| **`order_id`** | `VARCHAR` / `TEXT` | Identificador único do pedido. | Chave estrangeira de relacionamento. |
| **`order_item_id`** | `INTEGER` | Número sequencial do item dentro do mesmo pedido (Ex: 1, 2, 3). | Convertido para inteiro e usado na remoção de duplicados. |
| **`product_id`** | `VARCHAR` / `TEXT` | Identificador único do produto vendido. | Chave de relacionamento com o catálogo. |
| **`seller_id`** | `VARCHAR` / `TEXT` | Identificador único do vendedor. | Chave de relacionamento com a base de lojistas. |
| **`shipping_limit_date`** | `TIMESTAMP` | Data e hora limite para o envio do produto pelo vendedor. | **Tratada:** Convertida de texto bruto para o tipo `TIMESTAMP`. |
| **`price`** | `NUMERIC` / `FLOAT` | Preço de venda do produto em Reais (R$). | **Tratada:** Nulos limpos; valores cravados no teto de **R$ 277,40** via Winsorização. |
| **`freight_value`** | `NUMERIC` / `FLOAT` | Valor do frete do item em Reais (R$). | **Tratada:** Nulos limpos; valores cravados no teto de **R$ 33,40** via Winsorização. |

---

## 3. Qualidade dos Dados e Limpeza (Bronze $\rightarrow$ Silver)

Com base no seu script SQL de produção e nas validações executadas, os seguintes problemas de qualidade de dados foram mitigados:

* **Eliminação de Registros Duplicados:** Foi identificada a possibilidade de duplicidade técnica para a combinação de `order_id` e `order_item_id`. Aplicamos uma janela de partição (`ROW_NUMBER() OVER`) ordenada pela data máxima de envio (`shipping_limit_date DESC`) para garantir que apenas a última atualização de registro entrasse na Silver (`rank_duplicado = 1`).
* **Tratamento de Valores Inconsistentes:** Registros com preços zerados ou inválidos foram filtrados (`price_bruto > 0`), mantendo apenas transações comerciais válidas na base.

---

## 4. Análise Estatística Univariada e Tratamento de Outliers

Esta é a seção central do relatório, justificando as decisões técnicas tomadas sobre o comportamento de distribuição de **Preço** e **Frete**.

### 4.1. Análise de Distribuição e Teste de Normalidade

Ao aplicarmos o teste estatístico de **Shapiro-Wilk** sobre as amostras das variáveis financeiras, obtivemos os seguintes resultados:

* **Preço:** $p\text{-value} < 0{,}05$
* **Frete:** $p\text{-value} < 0{,}05$

**Conclusão Estatística:** Ambas as variáveis possuem distribuições **altamente assimétricas (cauda longa à direita)** e **NÃO seguem uma Distribuição Normal**. Isso significa que a maioria dos itens transacionados possui valores baixos a moderados, enquanto uma minoria de compras atinge valores muito expressivos (outliers).

### 4.2. Detecção e Justificativa de Outliers (Método IQR)

Utilizando a técnica da Amplitude Interquartílica ($IQR = Q3 - Q1$), mapeamos os limites matemáticos além dos quais os dados passam a distorcer análises preditivas lineares:

* **Preço ($price$):**
* $Q1$ (25% dos dados): R$ 39,90
* $Q3$ (75% dos dados): R$ 134,90
* **Limite Superior Teórico:** $Q3 + 1.5 \times IQR = \mathbf{R\$\,277{,}40}$


* **Frete ($freight\_value$):**
* $Q1$ (25% dos dados): R$ 13,08
* $Q3$ (75% dos dados): R$ 21,18
* **Limite Superior Teórico:** $Q3 + 1.5 \times IQR = \mathbf{R\$\,33{,}40}$



### 4.3. Estratégia de Engenharia de Recursos: Winsorização Padrão

Em vez de simplesmente deletar os registros acima desses tetos (o que destruiria o histórico transacional real do e-commerce e afetaria o cálculo de faturamento total na camada Gold), adotamos a **Winsorização Padrão**.

Criamos uma atualização direta (`UPDATE`) no banco Supabase para limitar os extremos aos valores exatos do IQR. O código em Python correspondente que documenta essa validação e plota a checagem é:

```python
# Célula de Validação Estatística no Notebook
import pandas as pd

# Carregando dados pós-tratamento da camada Silver
df_silver = pd.read_sql("SELECT price, freight_value FROM silver.order_items", engine)

print("📊 --- COMPROVAÇÃO DE SUCESSO DA WINSORIZAÇÃO ---")
print(f"Preço Máximo na Camada Silver: R$ {df_silver['price'].max():.2f} (Teto: R$ 277.40)")
print(f"Frete Máximo na Camada Silver: R$ {df_silver['freight_value'].max():.2f} (Teto: R$ 33.40)")

```

---

## 5. Conclusão e Próximos Passos (Conexão com a Camada Gold)

Com a tabela `silver.order_items` 100% higienizada e tratada contra anomalias extremas, atingimos os seguintes critérios de sucesso do Contrato de Dados:

1. **Idempotência Garantida:** Se a tabela quebrar, o script SQL reconstrói a estrutura de forma limpa, enquanto o script em Python garante a manutenção dos ajustes das colunas.
2. **Pronto para Modelagem:** O tratamento de outliers reduzirá o ruído em modelos de agrupamento (como K-Means para segmentação VIP) e modelos lineares de regressão.

**Próxima Etapa do Pipeline:** Realizar os `JOINs` e agregações com a tabela de `customers` (já analisada no relatório modelo anterior) para criar a tabela de características unificada na **Camada Gold**, dando início ao treinamento das IAs de Churn e Perfil de Clientes.